# Step 2 — FD001 Preprocessing and RUL Target Preparation

Objectives:

- create an engine-aware train/validation split
- generate raw RUL targets correctly
- define feature groups
- prevent data leakage
- identify preprocessing parameters using training engines only
- prepare data for later RUL model development

Important rule:
The official FD001 test set must not be used for preprocessing decisions,
hyperparameter tuning, feature-selection decisions, scaling parameters,
or abstention-threshold selection.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit

# Locate project root
cwd = Path.cwd()

if (cwd / "02_dataset").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "02_dataset").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate the project root.")

DATA_DIR = PROJECT_ROOT / "02_dataset" / "raw" / "CMAPSSData"

print("Project root:", PROJECT_ROOT)
print("Dataset directory:", DATA_DIR)
print("Dataset exists:", DATA_DIR.exists())

In [ ]:
column_names = (
    ["unit_id", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

train_fd001 = pd.read_csv(
    DATA_DIR / "train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=column_names
)

print("FD001 training shape:", train_fd001.shape)
print("Number of engines:", train_fd001["unit_id"].nunique())

In [ ]:
RANDOM_STATE = 42

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, val_idx = next(
    splitter.split(
        train_fd001,
        groups=train_fd001["unit_id"]
    )
)

dev_train = train_fd001.iloc[train_idx].copy()
validation = train_fd001.iloc[val_idx].copy()

dev_train = dev_train.sort_values(
    ["unit_id", "cycle"]
).reset_index(drop=True)

validation = validation.sort_values(
    ["unit_id", "cycle"]
).reset_index(drop=True)

In [ ]:
train_engines = set(dev_train["unit_id"].unique())
val_engines = set(validation["unit_id"].unique())

print("Development-train rows:", len(dev_train))
print("Validation rows       :", len(validation))

print("\nDevelopment-train engines:", len(train_engines))
print("Validation engines       :", len(val_engines))

print(
    "\nEngines appearing in both sets:",
    train_engines.intersection(val_engines)
)

print(
    "Total unique engines across both sets:",
    len(train_engines.union(val_engines))
)

In [ ]:
INTERIM_DIR = PROJECT_ROOT / "02_dataset" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

split_records = []

for unit_id in sorted(train_engines):
    split_records.append({
        "unit_id": unit_id,
        "split": "dev_train"
    })

for unit_id in sorted(val_engines):
    split_records.append({
        "unit_id": unit_id,
        "split": "validation"
    })

split_df = pd.DataFrame(split_records).sort_values("unit_id")

split_path = INTERIM_DIR / "fd001_engine_split_seed42.csv"
split_df.to_csv(split_path, index=False)

print("Split saved to:")
print(split_path)

display(split_df.head(10))

In [ ]:
def add_raw_rul(df):
    df = df.copy()

    max_cycle = df.groupby("unit_id")["cycle"].transform("max")
    df["RUL_raw"] = max_cycle - df["cycle"]

    return df


dev_train = add_raw_rul(dev_train)
validation = add_raw_rul(validation)

In [ ]:
print("Development-train RUL range:")
print(dev_train["RUL_raw"].min(), "to", dev_train["RUL_raw"].max())

print("\nValidation RUL range:")
print(validation["RUL_raw"].min(), "to", validation["RUL_raw"].max())

In [ ]:
dev_final_rows = (
    dev_train
    .groupby("unit_id")
    .tail(1)
)

val_final_rows = (
    validation
    .groupby("unit_id")
    .tail(1)
)

print(
    "Development-train engines ending with RUL != 0:",
    (dev_final_rows["RUL_raw"] != 0).sum()
)

print(
    "Validation engines ending with RUL != 0:",
    (val_final_rows["RUL_raw"] != 0).sum()
)

print(
    "Negative RUL values in development-train:",
    (dev_train["RUL_raw"] < 0).sum()
)

print(
    "Negative RUL values in validation:",
    (validation["RUL_raw"] < 0).sum()
)

In [ ]:
ID_COLUMNS = ["unit_id"]

TIME_COLUMNS = ["cycle"]

OPERATING_COLUMNS = [
    "op_setting_1",
    "op_setting_2",
    "op_setting_3"
]

SENSOR_COLUMNS = [
    f"sensor_{i}" for i in range(1, 22)
]

TARGET_COLUMN = "RUL_raw"

MEASUREMENT_COLUMNS = OPERATING_COLUMNS + SENSOR_COLUMNS

In [ ]:
print("Identifier columns :", ID_COLUMNS)
print("Time columns       :", TIME_COLUMNS)
print("Operating columns  :", len(OPERATING_COLUMNS))
print("Sensor columns     :", len(SENSOR_COLUMNS))
print("Measurement columns:", len(MEASUREMENT_COLUMNS))
print("Target column      :", TARGET_COLUMN)

print("\nTotal columns in dev_train:", len(dev_train.columns))

In [ ]:
constant_measurement_columns = [
    col
    for col in MEASUREMENT_COLUMNS
    if dev_train[col].nunique() == 1
]

print("Constant measurement columns based only on dev_train:")
print(constant_measurement_columns)

print(
    "\nNumber of constant measurement columns:",
    len(constant_measurement_columns)
)

In [ ]:
RETAINED_MEASUREMENT_COLUMNS = [
    col
    for col in MEASUREMENT_COLUMNS
    if col not in constant_measurement_columns
]

print("Original measurement columns :", len(MEASUREMENT_COLUMNS))
print("Constant columns removed      :", len(constant_measurement_columns))
print("Retained measurement columns  :", len(RETAINED_MEASUREMENT_COLUMNS))

print("\nRetained columns:")
print(RETAINED_MEASUREMENT_COLUMNS)

In [ ]:
assert len(RETAINED_MEASUREMENT_COLUMNS) == 17
assert not any(
    col in RETAINED_MEASUREMENT_COLUMNS
    for col in constant_measurement_columns
)

print("Feature-list consistency check passed.")

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Learn scaling parameters ONLY from development-training data
scaler.fit(
    dev_train[RETAINED_MEASUREMENT_COLUMNS]
)

# Keep original dataframes unchanged
dev_train_scaled = dev_train.copy()
validation_scaled = validation.copy()

# Apply the same learned transformation
dev_train_scaled[RETAINED_MEASUREMENT_COLUMNS] = scaler.transform(
    dev_train[RETAINED_MEASUREMENT_COLUMNS]
)

validation_scaled[RETAINED_MEASUREMENT_COLUMNS] = scaler.transform(
    validation[RETAINED_MEASUREMENT_COLUMNS]
)

print("Scaling completed.")

In [ ]:
max_abs_mean = (
    dev_train_scaled[RETAINED_MEASUREMENT_COLUMNS]
    .mean()
    .abs()
    .max()
)

population_std = pd.Series(
    np.std(
        dev_train_scaled[RETAINED_MEASUREMENT_COLUMNS],
        axis=0,
        ddof=0
    ),
    index=RETAINED_MEASUREMENT_COLUMNS
)

max_std_deviation = (
    population_std - 1.0
).abs().max()

print("Largest absolute scaled mean:", max_abs_mean)
print("Largest deviation of scaled std from 1:", max_std_deviation)

In [ ]:
print(
    "Missing values after scaling - dev_train:",
    dev_train_scaled[RETAINED_MEASUREMENT_COLUMNS].isna().sum().sum()
)

print(
    "Missing values after scaling - validation:",
    validation_scaled[RETAINED_MEASUREMENT_COLUMNS].isna().sum().sum()
)

print(
    "Infinite values after scaling - dev_train:",
    np.isinf(
        dev_train_scaled[RETAINED_MEASUREMENT_COLUMNS].to_numpy()
    ).sum()
)

print(
    "Infinite values after scaling - validation:",
    np.isinf(
        validation_scaled[RETAINED_MEASUREMENT_COLUMNS].to_numpy()
    ).sum()
)

In [ ]:
print("Development-train RUL statistics:")
display(
    dev_train["RUL_raw"].describe()
)

print("\nValidation RUL statistics:")
display(
    validation["RUL_raw"].describe()
)

In [ ]:
rul_thresholds = [100, 125, 130, 150]

for threshold in rul_thresholds:
    dev_count = (dev_train["RUL_raw"] > threshold).sum()
    dev_pct = 100 * dev_count / len(dev_train)

    val_count = (validation["RUL_raw"] > threshold).sum()
    val_pct = 100 * val_count / len(validation)

    print(
        f"RUL > {threshold:3d} | "
        f"dev_train: {dev_count:5d} ({dev_pct:5.1f}%) | "
        f"validation: {val_count:4d} ({val_pct:5.1f}%)"
    )

In [ ]:
RUL_CAP_CANDIDATES = [125, 130]

for cap in RUL_CAP_CANDIDATES:
    dev_train[f"RUL_cap_{cap}"] = np.minimum(
        dev_train["RUL_raw"],
        cap
    )

    validation[f"RUL_cap_{cap}"] = np.minimum(
        validation["RUL_raw"],
        cap
    )

print(
    dev_train[
        ["RUL_raw", "RUL_cap_125", "RUL_cap_130"]
    ].head()
)

In [ ]:
for cap in RUL_CAP_CANDIDATES:
    print(
        f"Cap {cap}: "
        f"dev max = {dev_train[f'RUL_cap_{cap}'].max()}, "
        f"validation max = {validation[f'RUL_cap_{cap}'].max()}"
    )

    assert (
        dev_train[f"RUL_cap_{cap}"]
        <= dev_train["RUL_raw"]
    ).all()

    assert (
        validation[f"RUL_cap_{cap}"]
        <= validation["RUL_raw"]
    ).all()

print("Candidate capped-RUL checks passed.")

In [ ]:
PRIMARY_INPUT_COLUMNS = RETAINED_MEASUREMENT_COLUMNS.copy()

CYCLE_AUGMENTED_INPUT_COLUMNS = (
    ["cycle"] + RETAINED_MEASUREMENT_COLUMNS
)

print("Primary model input features:",
      len(PRIMARY_INPUT_COLUMNS))

print("Cycle-augmented input features:",
      len(CYCLE_AUGMENTED_INPUT_COLUMNS))

print("\nPrimary input columns:")
print(PRIMARY_INPUT_COLUMNS)

In [ ]:
assert "cycle" not in PRIMARY_INPUT_COLUMNS
assert "unit_id" not in PRIMARY_INPUT_COLUMNS
assert "RUL_raw" not in PRIMARY_INPUT_COLUMNS

assert "cycle" in CYCLE_AUGMENTED_INPUT_COLUMNS

print("Input-feature role checks passed.")

### Cycle feature decision

The `cycle` variable is retained for temporal ordering, window construction,
RUL computation, and diagnostic analysis, but it is excluded from the primary
RUL predictor.

Reason:
The study focuses on RUL reliability under sensor degradation. Including an
uncorrupted cycle/age variable in the main predictor may encourage an
age-based shortcut and partially mask the effect of degraded sensor
measurements.

A cycle-augmented model may later be evaluated as an ablation/sensitivity
experiment.

In [ ]:
TARGET_COLUMNS = [
    "RUL_raw",
    "RUL_cap_125",
    "RUL_cap_130"
]

# Metadata + targets
dev_model_df = dev_train[
    ["unit_id", "cycle"] + TARGET_COLUMNS
].copy()

val_model_df = validation[
    ["unit_id", "cycle"] + TARGET_COLUMNS
].copy()

# Add the already-scaled primary input features
dev_model_df[PRIMARY_INPUT_COLUMNS] = (
    dev_train_scaled[PRIMARY_INPUT_COLUMNS]
)

val_model_df[PRIMARY_INPUT_COLUMNS] = (
    validation_scaled[PRIMARY_INPUT_COLUMNS]
)

print("Development model-ready shape:", dev_model_df.shape)
print("Validation model-ready shape :", val_model_df.shape)

In [ ]:
assert len(dev_model_df) == len(dev_train)
assert len(val_model_df) == len(validation)

assert dev_model_df.isna().sum().sum() == 0
assert val_model_df.isna().sum().sum() == 0

assert not np.isinf(
    dev_model_df[PRIMARY_INPUT_COLUMNS].to_numpy()
).any()

assert not np.isinf(
    val_model_df[PRIMARY_INPUT_COLUMNS].to_numpy()
).any()

assert dev_model_df["unit_id"].equals(dev_train["unit_id"])
assert val_model_df["unit_id"].equals(validation["unit_id"])

assert dev_model_df["cycle"].equals(dev_train["cycle"])
assert val_model_df["cycle"].equals(validation["cycle"])

print("Model-ready dataset integrity checks passed.")

In [ ]:
import json
import joblib

PROCESSED_DIR = (
    PROJECT_ROOT
    / "02_dataset"
    / "processed"
    / "FD001"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

dev_model_df.to_csv(
    PROCESSED_DIR / "fd001_dev_train_scaled.csv",
    index=False
)

val_model_df.to_csv(
    PROCESSED_DIR / "fd001_validation_scaled.csv",
    index=False
)

joblib.dump(
    scaler,
    PROCESSED_DIR / "fd001_standard_scaler.joblib"
)

print("Processed artifacts saved to:")
print(PROCESSED_DIR)

In [ ]:
preprocessing_config = {
    "dataset": "C-MAPSS FD001",
    "random_state": RANDOM_STATE,
    "validation_fraction_engines": 0.20,

    "constant_measurement_columns":
        constant_measurement_columns,

    "primary_input_columns":
        PRIMARY_INPUT_COLUMNS,

    "cycle_augmented_input_columns":
        CYCLE_AUGMENTED_INPUT_COLUMNS,

    "canonical_target":
        "RUL_raw",

    "candidate_capped_targets": {
        "125": "RUL_cap_125",
        "130": "RUL_cap_130"
    },

    "scaling_method":
        "StandardScaler",

    "scaler_fit_on":
        "development-training engines only"
}

with open(
    PROCESSED_DIR / "fd001_preprocessing_config.json",
    "w"
) as f:
    json.dump(
        preprocessing_config,
        f,
        indent=4
    )

print("Preprocessing configuration saved.")

In [ ]:
print("Saved FD001 preprocessing files:\n")

for file_path in sorted(PROCESSED_DIR.iterdir()):
    print(file_path.name)

In [ ]:
# Reload saved datasets
dev_reloaded = pd.read_csv(
    PROCESSED_DIR / "fd001_dev_train_scaled.csv"
)

val_reloaded = pd.read_csv(
    PROCESSED_DIR / "fd001_validation_scaled.csv"
)

# Reload scaler
scaler_reloaded = joblib.load(
    PROCESSED_DIR / "fd001_standard_scaler.joblib"
)

# Reload preprocessing configuration
with open(
    PROCESSED_DIR / "fd001_preprocessing_config.json",
    "r"
) as f:
    config_reloaded = json.load(f)


print("Reloaded development shape:", dev_reloaded.shape)
print("Reloaded validation shape :", val_reloaded.shape)

print("\nScaler feature count:",
      len(scaler_reloaded.mean_))

print("Config dataset:",
      config_reloaded["dataset"])

print("Config canonical target:",
      config_reloaded["canonical_target"])

print("Config primary feature count:",
      len(config_reloaded["primary_input_columns"]))

In [ ]:
# Structural consistency
assert dev_reloaded.shape == dev_model_df.shape
assert val_reloaded.shape == val_model_df.shape

assert list(dev_reloaded.columns) == list(dev_model_df.columns)
assert list(val_reloaded.columns) == list(val_model_df.columns)

assert (
    config_reloaded["primary_input_columns"]
    == PRIMARY_INPUT_COLUMNS
)

assert (
    len(scaler_reloaded.mean_)
    == len(PRIMARY_INPUT_COLUMNS)
)

# Check saved numerical values survived CSV round-trip
assert np.allclose(
    dev_reloaded[PRIMARY_INPUT_COLUMNS].to_numpy(),
    dev_model_df[PRIMARY_INPUT_COLUMNS].to_numpy(),
    rtol=1e-10,
    atol=1e-12
)

assert np.allclose(
    val_reloaded[PRIMARY_INPUT_COLUMNS].to_numpy(),
    val_model_df[PRIMARY_INPUT_COLUMNS].to_numpy(),
    rtol=1e-10,
    atol=1e-12
)

print("Reload-and-reproducibility checks passed.")

In [ ]:
split_file = (
    PROJECT_ROOT
    / "02_dataset"
    / "interim"
    / "fd001_engine_split_seed42.csv"
)

print("Frozen engine split exists:", split_file.exists())